# Wood Design - Implementation Notes

Unit definitions and calculation templates for wood structural design.
Theory and design procedures reference:

- Breyer, D. et al. *Design of Wood Structures -- ASD/LRFD*, 8th Ed. McGraw Hill.

Refer to the original text and the NDS (National Design Specification) for
design values and procedures.


# Preface, Publications and Units

Most of the contents of the book won't be captured in this document, and  
many theoretical sections will be glossed over. I highly recommend any  
readers obtain a copy of the original source material so that that  
information contained within isn't lost to them.

The main library used in this notebook is Civilpy, it can be installed in a  
code cell with;

`%pip install Civilpy`

In [2]:
# This code checks to make sure civilpy is installed and verifies the version
%pip show civilpy

## Units

Since this document utilizes python, the [pint](https://pint.readthedocs.io/en/stable/) library has been integrated  
into the code to ensure unit handling is done during all practice problems  
where practicable.

If pint is not installed in your environment already, you can run the  
following in a jupyter code cell to get started. If using civilpy, it's  
included as a dependency.

`%pip install pint%`

In [2]:
from civilpy.general import units

The following are examples of how to assign each type of unit when doing  
calculations, how to convert them, and how they output in the jupyter view.

In [3]:
# You can set pint to default to imperial during conversions,
units.default_system = 'imperial'

In [4]:
# ft, foot, feet
foot = 1 * units('ft')
foot

In [5]:
# Note that some units are also accessible as attributes of the unit class,
foot = 1 * units.foot
foot  # Throughout the document though, function calls are the preferred method

In [6]:
# ft^2, square foot, square feet
sq_ft = 1 * units('ft^2')
sq_ft

In [7]:
# in., inch, inches
inch = 1 * units('in')
inch

In [8]:
# in.^2, square inch, square inches
sq_in = 1 * units('in^2')
sq_in

In [9]:
# Also note that the formatting changes if you call print on a pint unit
print(sq_in)
sq_in

In [10]:
# Or HTML which notebooks are able to render in their output,
units.formatter.default_format = "H"
print(sq_in)

In [11]:
# You can change the default behavior of pint to always use the 'pretty' output
units.formatter.default_format = "P"
print(sq_in)

In [12]:
# k, 1000 lbs, (kip, kilopound)
kip = 1 * units('kip')
kip

In [13]:
# ksi, kips per square inch (k/in^2)
ksi = 1 * units('kip/in^2')
ksi

In [14]:
# Converting units are easy once defined in pint, note floating point errors
print(ksi.to('lbf/in^2'))

# You can fix/account for floating point errors by rounding
round(ksi.to('lbf/in^2'), 2)

In [15]:
# mph, miles per hour
mph = 1 * units('mph')
mph

In [16]:
# pcf, pounds per cubic foot (lb/ft^3)
pcf = units('lbf/ft^3')
pcf

In [17]:
# plf, pounds per lineal foot (lb/ft)
plf = units('lbf/ft')
plf

In [18]:
# psf, pounds per square foot (lb/ft^2)
psf = units('lbf/ft^2')
psf

In [19]:
# psi, pounds per square inch (lb/in^2)
psi = units('lbf/in^2')
psi

In [20]:
# sec, second (sec)
sec = units('sec')
sec

## 1.7 Structural Calculations

The book mentions that equation solving software or spreadsheets can be  
setup as templates to make generating solutions easier. This notebook can  
be utilized as a template, but it's recommended to pull the relevant  
calculations into a new notebook as a template to streamline the output  
calculations and not carry forward the background information provided by  
these markdown cells.

When equations are provided, they are provided first in standard equation  
notation via markdown. Then they are solved by code cells below by defining  
relevant python functions and utilizing them to solve the problem. In the  
book the conversion from lbs to kips is done without formal notation. This  
can't be done in code since it requires a user to specify if they want the  
result in a different format. The following is the example demonstration of  
how the formulas are provided throughout the book.

$$ 
\begin{aligned} 
    T &= F'_t A \\ &= (1200\ lb/in.^2)(20\ in.^2) \\ &= 24\ k 
\end{aligned}
$$

Where 

$T$ = tensile force
$F'_t$ = adjusted tensile design value
$A$ = cross-sectional area

Here is how the above would be performed in python,

In [26]:
tensile_force = 1200 * units('lbf/in^2') * 20 * units('in^2')
round(tensile_force.to('kips'), 0)

if you want to hold the design values in additional variables,

In [31]:
tensile_design_force = 1200 * units('lbf/in^2')
cross_sectional_area = 20 * units('in^2')

tensile_force = round(
    (tensile_design_force * cross_sectional_area).to('kips'), 
    0
)

tensile_force

# 2. Reference Design Values

The `civilpy.structural.wood` module provides section property lookups for
sawn lumber and glulam timber with NDS reference design values automatically
attached. All values carry Pint units for dimensional consistency.

## 2.1 Sawn Lumber Sections

`LumberSection(nominal_width, nominal_depth, species, grade)` looks up:
- **Actual dimensions** from NDS dressed lumber sizes
- **Section properties**: A, Sx, Ix, Sy, Iy, rx, ry
- **Reference design values**: Fb, Ft, Fv, Fc_perp, Fc, E, Emin
- **Wet service factors** (CM) per property

In [ ]:
from civilpy.structural.wood import (
    LumberSection, GlulamSection, AdjustmentFactors, WoodMemberCheck,
    TimberBridgeDeck, TimberStringer, BoltConnection,
    euler_critical_buckling_stress, column_stability_factor,
    beam_stability_factor, effective_beam_length,
    list_available_species, list_available_grades, list_available_glulam,
    LOAD_DURATION_FACTORS, TIMBER_BRIDGE_LOAD_COMBOS, WEARING_SURFACE_WEIGHTS,
)
from civilpy.general import units
import pandas as pd

# Set default to imperial and pretty formatting
units.default_system = 'imperial'
units.formatter.default_format = "P"

In [ ]:
# View available species and grades in the database
print("Available species:")
for s in list_available_species():
    print(f"  {s}")
print("\nAvailable grades for Douglas Fir-Larch:")
for g in list_available_grades("Douglas Fir-Larch"):
    print(f"  {g}")

In [ ]:
# Create a sawn lumber section and inspect properties
stringer = LumberSection(8, 16, "Douglas Fir-Larch", "Select Structural")
print(f"Section: {stringer}")
print(f"  Actual dimensions: b = {stringer.b}, d = {stringer.d}")
print(f"  Area:     A  = {stringer.A}")
print(f"  Sx:       Sx = {stringer.Sx}")
print(f"  Ix:       Ix = {stringer.Ix}")
print(f"  Weight:   w  = {stringer.weight:.2f}")
print(f"\nReference Design Values (NDS Supplement Table 4D):")
print(f"  Fb    = {stringer.Fb}")
print(f"  Ft    = {stringer.Ft}")
print(f"  Fv    = {stringer.Fv}")
print(f"  Fc_perp = {stringer.Fc_perp}")
print(f"  Fc    = {stringer.Fc}")
print(f"  E     = {stringer.E}")
print(f"  Emin  = {stringer.Emin}")
print(f"  G     = {stringer.specific_gravity}")

## 2.2 Glulam Sections

`GlulamSection(width, depth, combination, species)` provides the same
interface but with glulam-specific properties:
- **Fb_pos / Fb_neg**: Positive and negative face bending values
- Standard glulam widths: 3.125, 5.125, 6.75, 8.75, 10.75 in.

In [ ]:
# Create a glulam section
glulam_beam = GlulamSection(6.75, 30, "24F-V4", "Douglas Fir")
print(f"Section: {glulam_beam}")
print(f"  b = {glulam_beam.b}, d = {glulam_beam.d}")
print(f"  A  = {glulam_beam.A}")
print(f"  Sx = {glulam_beam.Sx}")
print(f"  Ix = {glulam_beam.Ix}")
print(f"\nReference Design Values (NDS Supplement Table 5A):")
print(f"  Fb+   = {glulam_beam.Fb_pos}")
print(f"  Fb-   = {glulam_beam.Fb_neg}")
print(f"  Ft    = {glulam_beam.Ft}")
print(f"  Fv    = {glulam_beam.Fv}")
print(f"  Fc    = {glulam_beam.Fc}")
print(f"  E     = {glulam_beam.E}")
print(f"  Emin  = {glulam_beam.Emin}")

# View all available glulam combinations
print("\nAvailable Glulam Combinations:")
print(list_available_glulam())

# 3. NDS Adjustment Factors

The NDS requires reference design values to be multiplied by a series of
adjustment factors to obtain the **adjusted design value** $F'$:

$$F'_b = F_b \times C_D \times C_M \times C_t \times C_L \times C_F \times C_i \times C_r$$

| Factor | Description | NDS Reference |
|--------|------------|---------------|
| $C_D$ | Load duration | Table 2.3.2 |
| $C_M$ | Wet service | Table 4.3.3 / 5.3.3 |
| $C_t$ | Temperature | Table 2.3.3 |
| $C_L$ | Beam stability (lateral-torsional buckling) | 3.3.3 |
| $C_F$ | Size factor (sawn lumber) | Table 4.3.1 |
| $C_i$ | Incising factor (preservative treatment) | Table 4.3.8 |
| $C_r$ | Repetitive member (1.15) | 4.3.4 |
| $C_P$ | Column stability (compression) | 3.7.1 |
| $C_b$ | Bearing area | 3.10.4 |
| $C_V$ | Volume factor (glulam) | 5.3.6 |

The `AdjustmentFactors` class computes each factor individually and
the `adjusted_value()` method applies all applicable factors at once.

In [ ]:
# Demonstrate individual adjustment factors
beam = LumberSection(6, 12, "Douglas Fir-Larch", "No. 1")
af = AdjustmentFactors(beam, method="ASD")

print("Load Duration Factors (CD) per NDS Table 2.3.2:")
for name, val in LOAD_DURATION_FACTORS.items():
    print(f"  {name:12s}: CD = {val}")

print(f"\nWet Service Factors (CM) for {beam.species}:")
for prop in ["Fb", "Ft", "Fv", "Fc_perp", "Fc", "E", "Emin"]:
    cm_dry = af.CM(prop, wet_service=False)
    cm_wet = af.CM(prop, wet_service=True)
    print(f"  {prop:8s}: dry={cm_dry:.3f}, wet={cm_wet:.3f}")

print(f"\nSize Factors (CF) for {beam.nominal_width}x{beam.nominal_depth}:")
for prop in ["Fb", "Ft", "Fc"]:
    print(f"  CF({prop}) = {af.CF(prop)}")

print(f"\nIncising Factors (Ci) -- when incised for treatment:")
for prop in ["Fb", "Ft", "Fv", "Fc_perp", "Fc", "E", "Emin"]:
    print(f"  Ci({prop:8s}) = {af.Ci(prop, incised=True)}")

print(f"\nRepetitive Member Factor: Cr = {af.Cr(repetitive=True)}")

In [ ]:
# Compute fully adjusted Fb' for a bridge stringer scenario:
#   - Wet service (exposed bridge, MC > 19%)
#   - Incised for preservative treatment (ACQ)
#   - Normal load duration
#   - Fully braced (CL = 1.0)

result = af.adjusted_value(
    "Fb",
    load_duration="normal",
    wet_service=True,
    temperature_f=100.0,
    incised=True,
    repetitive=False,
    lu=None,  # Fully braced by deck
)

print(f"Reference Fb = {beam.Fb}")
print(f"Adjustment Factors Applied:")
for name, val in result["factors"].items():
    print(f"  {name:6s} = {val:.3f}")
print(f"\nAdjusted Fb' = {result['value']:.1f}")

# 4. Sawn Lumber Beam Design Example

**Problem**: Design check a 6x12 Douglas Fir-Larch No. 1 beam spanning 16 ft
with a uniform dead load of 100 plf and live load of 200 plf.
The beam is fully braced by decking. Dry service, not incised.

$$f_b = \frac{M}{S_x} \leq F'_b$$

$$f_v = \frac{3V}{2A} \leq F'_v$$

$$\Delta = \frac{5wL^4}{384E'I} \leq \frac{L}{360}$$

In [ ]:
# Define section and loads
beam = LumberSection(6, 12, "Douglas Fir-Larch", "No. 1")
span = 16 * units("ft")
w_dead = 100 * units("lbf/ft")
w_live = 200 * units("lbf/ft")
w_total = w_dead + w_live

# Compute demands
M = (w_total * span**2 / 8).to("lbf*ft")
V = (w_total * span / 2).to("lbf")

print(f"Section: {beam}")
print(f"Span: {span}")
print(f"Total load: w = {w_total}")
print(f"Max moment: M = {M:.0f}")
print(f"Max shear:  V = {V:.0f}")

# Run full member check (dry service, not incised, normal duration)
check = WoodMemberCheck(beam, method="ASD", wet_service=False,
                         incised=False, load_duration="normal")

bending = check.check_bending(M)
shear = check.check_shear(V)

# Deflection check uses live load only per convention
defl = check.check_deflection(w_live, span, limit="L/360")

# Bearing check at support (6 in. bearing length)
R = V  # Reaction = max shear for simple span
bearing = check.check_bearing(R, 6 * units("in"))

print("\n--- Results ---")
print(f"Bending: fb = {bending['fb']:.1f}, Fb' = {bending['Fb_prime']:.1f}, "
      f"D/C = {bending['ratio']} [{bending['status']}]")
print(f"Shear:   fv = {shear['fv']:.1f}, Fv' = {shear['Fv_prime']:.1f}, "
      f"D/C = {shear['ratio']} [{shear['status']}]")
print(f"Defl:    delta = {defl['delta']:.3f}, limit = {defl['delta_limit']:.3f}, "
      f"D/C = {defl['ratio']} [{defl['status']}]")
print(f"Bearing: fc_perp = {bearing['fc_perp']:.1f}, Fc_perp' = {bearing['Fc_perp_prime']:.1f}, "
      f"D/C = {bearing['ratio']} [{bearing['status']}]")

print("\n--- Summary Table ---")
check.summary()

# 5. Column Design Example

**Problem**: Check a 6x6 Douglas Fir-Larch No. 1 column, 10 ft tall, carrying
an axial load of 15,000 lbs. Pin-pin end conditions (K=1.0). Dry service.

The column stability factor $C_P$ accounts for buckling:

$$F_{cE} = \frac{0.822 \cdot E'_{min}}{(l_e/d)^2}$$

$$C_P = \frac{1 + F_{cE}/F_c^*}{2c} - \sqrt{\left(\frac{1 + F_{cE}/F_c^*}{2c}\right)^2 - \frac{F_{cE}/F_c^*}{c}}$$

where $c = 0.8$ for sawn lumber, $c = 0.9$ for glulam.

In [ ]:
# Column design check
col = LumberSection(6, 6, "Douglas Fir-Larch", "No. 1")
P = 15000 * units("lbf")
L_col = 10 * units("ft")
K = 1.0
le = K * L_col

print(f"Section: {col}")
print(f"  b = {col.b}, d = {col.d}, A = {col.A}")
print(f"Applied load: P = {P}")
print(f"Effective length: le = {le}")

check_col = WoodMemberCheck(col, method="ASD", wet_service=False,
                             incised=False, load_duration="normal")
comp = check_col.check_compression(P, le)

print(f"\nResults:")
print(f"  fc     = P/A = {comp['fc']:.1f}")
print(f"  le/d (strong) = {comp['le_d_strong']}")
print(f"  le/d (weak)   = {comp['le_d_weak']}")
print(f"  Cp     = {comp['Cp']:.3f}")
print(f"  Fc'    = {comp['Fc_prime']:.1f}")
print(f"  D/C    = {comp['ratio']} [{comp['status']}]")

# Show the Euler critical stress explicitly
FcE = euler_critical_buckling_stress(col.Emin, le, col.b)
print(f"\n  FcE = 0.822 * Emin / (le/d)^2 = {FcE:.1f}")

# 6. Glulam Beam Design

**Problem**: Check a 6-3/4 x 30 in. Douglas Fir 24F-V4 glulam beam spanning
40 ft. Uniform dead load of 200 plf and live load of 500 plf.
Dry service. Fully braced by deck.

For glulam, the **Volume Factor** $C_V$ replaces the Size Factor:

$$C_V = \left(\frac{21}{L}\right)^{1/x} \left(\frac{12}{d}\right)^{1/x} \left(\frac{5.125}{b}\right)^{1/x}$$

where $x = 10$ and $L$ is span in feet, $d$ is depth in inches, $b$ is width in inches.

Per NDS 5.3.6, use the **lesser** of $C_L$ and $C_V$, not both.

In [ ]:
# Glulam beam check
gl = GlulamSection(6.75, 30, "24F-V4", "Douglas Fir")
span_gl = 40 * units("ft")
w_dl = 200 * units("lbf/ft")
w_ll = 500 * units("lbf/ft")
w_total_gl = w_dl + w_ll

M_gl = (w_total_gl * span_gl**2 / 8).to("lbf*ft")
V_gl = (w_total_gl * span_gl / 2).to("lbf")

print(f"Section: {gl}")
print(f"Span: {span_gl}, Total load: {w_total_gl}")
print(f"Max M = {M_gl:.0f}, Max V = {V_gl:.0f}")

# Show volume factor
af_gl = AdjustmentFactors(gl)
CV = af_gl.CV(L=span_gl)
print(f"\nVolume Factor CV = {CV:.4f}")

# Full check
check_gl = WoodMemberCheck(gl, method="ASD")
bend_gl = check_gl.check_bending(M_gl, span_length=span_gl)
shear_gl = check_gl.check_shear(V_gl)
defl_gl = check_gl.check_deflection(w_ll, span_gl, limit="L/360")
bearing_gl = check_gl.check_bearing(V_gl, 8 * units("in"), gl.b)

print(f"\n--- Summary ---")
check_gl.summary()

# 7. Timber Bridge Deck Design

**Problem**: Check a nail-laminated timber bridge deck spanning 4 ft between
stringers. 4x12 Douglas Fir-Larch No. 2 laminations on edge.
3 in. asphalt wearing surface. HL-93 loading per AASHTO.

The `TimberBridgeDeck` class automates:
- Wheel load distribution width per AASHTO 4.6.2.1.3
- Dead load computation (self-weight + wearing surface)
- Strength I factored demands (1.25*DC + 1.75*LL*(1+IM))
- Bending, shear, and deflection checks on a 1-ft-wide strip

In [ ]:
# Timber bridge deck check
deck = TimberBridgeDeck(
    deck_type="nail_laminated",
    span=4 * units("ft"),
    thickness=12 * units("in"),  # 4x12 laminations on edge, depth = 11.25"
    species="Douglas Fir-Larch",
    grade="No. 2",
    wearing_surface="asphalt_3in",
)

print(f"Deck type: {deck.deck_type}")
print(f"Span: {deck.span}")
print(f"Wearing surface: {deck.ws_weight}")

E_dist = deck.wheel_load_distribution_width()
print(f"Wheel load distribution width: E = {E_dist:.1f}")

w_dl = deck.dead_load_per_ft()
print(f"Dead load per ft width: {w_dl:.1f}")

result = deck.check_deck()
print(f"\n--- Deck Check Results (Strength I) ---")
print(f"Distribution width: {result['distribution_width']:.1f}")

b = result["bending"]
print(f"\nBending: fb = {b['fb']:.1f}, Fb' = {b['Fb_prime']:.1f}, "
      f"D/C = {b['ratio']} [{b['status']}]")

s = result["shear"]
print(f"Shear:   fv = {s['fv']:.1f}, Fv' = {s['Fv_prime']:.1f}, "
      f"D/C = {s['ratio']} [{s['status']}]")

d = result["deflection"]
print(f"Defl:    delta = {d['delta']:.3f}, limit = {d['delta_limit']:.3f}, "
      f"D/C = {d['ratio']} [{d['status']}]")

# 8. Timber Bridge Stringer Rating

**Problem**: Rate a timber bridge stringer for HL-93 loading per AASHTO MBE.

Bridge configuration:
- Span: 20 ft simple span
- 5 stringers at 4 ft spacing  
- 8x16 Douglas Fir-Larch Select Structural
- Nail-laminated deck, 3 in. asphalt wearing surface
- Wet service (exposed), incised (ACQ treatment)

The AASHTO MBE rating factor equation:

$$RF = \frac{\phi_c \phi_s C - \gamma_{DC} \cdot DC - \gamma_{DW} \cdot DW}{\gamma_{LL} \cdot LL \cdot (1 + IM)}$$

where:
- $C$ = member capacity ($F'_b \times S_x$)
- $DC$ = dead load effect
- $DW$ = wearing surface effect
- $LL$ = live load effect per stringer (with distribution factor $g$)
- $IM$ = dynamic load allowance (0.33)

In [ ]:
# ================================================================
# Timber Bridge Stringer Rating Example
# ================================================================

# --- Section & Geometry ---
stringer = LumberSection(8, 16, "Douglas Fir-Larch", "Select Structural")
span = 20 * units("ft")
spacing = 4 * units("ft")
num_stringers = 5

print(f"Stringer: {stringer}")
print(f"Span = {span}, Spacing = {spacing}")

# --- Dead Loads ---
# Stringer self-weight
w_stringer = stringer.weight
# Deck tributary: 4x12 lams @ 50 pcf, 4 ft spacing
deck_thickness = 12 * units("in")
w_deck = (deck_thickness.to("ft") * spacing * 50 * units("lbf/ft^3")).to("lbf/ft")
# Total DC
w_DC = w_stringer + w_deck
print(f"\nDead loads:")
print(f"  Stringer self-weight: {w_stringer:.1f}")
print(f"  Deck tributary:       {w_deck:.1f}")
print(f"  Total DC:             {w_DC:.1f}")

# Wearing surface (DW): 3 in. asphalt over 4 ft tributary
w_DW = (WEARING_SURFACE_WEIGHTS["asphalt_3in"] * spacing).to("lbf/ft")
print(f"  DW (asphalt):         {w_DW:.1f}")

# --- Dead Load Effects ---
M_DC = (w_DC * span**2 / 8).to("lbf*ft")
M_DW = (w_DW * span**2 / 8).to("lbf*ft")
print(f"\nDead Load Moments:")
print(f"  M_DC = {M_DC:.0f}")
print(f"  M_DW = {M_DW:.0f}")

# --- Live Load Distribution ---
ts = TimberStringer(stringer, span, spacing, num_lanes=1,
                     deck_type="nail_laminated")
g = ts.live_load_distribution_factor(num_stringers)
IM = ts.dynamic_load_allowance()
print(f"\nLive Load Distribution:")
print(f"  g  = S/6.0 = {spacing.to('ft').magnitude}/6.0 = {g} lanes/beam")
print(f"  IM = {IM}")

# HL-93 design truck moment (simple span, 20 ft)
# For a 20 ft span, governing is HL-93 truck: M_LL ~ 200 k-ft per lane
# (Approximate using simple beam formula for AASHTO design truck)
# Axle loads: 8k + 32k + 32k, spacings 14ft + 14-30ft
# For 20 ft span, critical is 32k axle at midspan + 8k off span
M_LL_lane = 160 * units("kip*ft")  # Approximate for 20 ft span
M_LL_stringer = (g * M_LL_lane).to("lbf*ft")
print(f"  M_LL (per lane) = {M_LL_lane}")
print(f"  M_LL (per stringer) = {M_LL_stringer:.0f}")

# --- Member Capacity ---
check = WoodMemberCheck(stringer, method="ASD", wet_service=True,
                         incised=True, load_duration="normal")
adj = check.af.adjusted_value("Fb", "normal", wet_service=True,
                               incised=True)
Fb_prime = adj["value"]
capacity_moment = (Fb_prime * stringer.Sx).to("lbf*ft")
print(f"\nMember Capacity:")
print(f"  Fb' = {Fb_prime:.1f}")
print(f"  Factors: {adj['factors']}")
print(f"  M_capacity = Fb' * Sx = {capacity_moment:.0f}")

# --- Rating Factor ---
rf = ts.rating_factor(
    DC_moment=M_DC,
    DW_moment=M_DW,
    LL_moment=M_LL_stringer,
    capacity_moment=capacity_moment,
    gamma_DC=1.25,
    gamma_DW=1.50,
    gamma_LL=1.75,
    phi_c=1.0,  # Good condition
    phi_s=1.0,
)

print(f"\n=== RATING FACTOR (Strength I) ===")
print(f"  RF = {rf['RF']}")
print(f"  Status: {rf['status']}")
print(f"\nLoad combination: gamma_DC={1.25}, gamma_DW={1.50}, gamma_LL={1.75}")
print(f"IM = {rf['IM']}")

# 9. Connection Design -- Bolted Stringer-to-Cap

**Problem**: Design a bolted connection of a timber stringer to a cap beam.
- 3/4 in. A307 bolts in single shear
- Main member: 8x16 Douglas Fir-Larch (G = 0.50)
- Side member: 3/8 in. steel plate
- 4 bolts in a single row

The European Yield Model (EYM) per NDS Chapter 12 evaluates six yield
modes (Im, Is, II, IIIm, IIIs, IV) and the governing (lowest) value
is the reference lateral design value $Z$.

$$Z' = Z \times C_D \times C_M \times C_t \times C_g \times C_\Delta$$

In [ ]:
# Bolted connection design
conn = BoltConnection(
    bolt_diameter=0.75 * units("in"),
    main_thickness=7.5 * units("in"),     # 8x16 actual width
    side_thickness=0.375 * units("in"),   # 3/8" steel plate
    main_species_gravity=0.50,
    side_material="steel",
    loading="single_shear",
    angle_to_grain=0,  # Load parallel to grain
)

# Reference single-bolt lateral value
z_ref = conn.Z_reference()
print(f"Bolt: 3/4 in. A307")
print(f"Main member: 8x16 DF-L (G = {conn.Gm}), lm = {conn.lm} in.")
print(f"Side member: 3/8 in. steel plate")
print(f"\nDowel bearing strengths:")
print(f"  Fem (wood) = {conn.Fem:.0f} psi")
print(f"  Fes (steel) = {conn.Fes:.0f} psi")
print(f"\nEYM Yield Mode Values:")
for mode, val in z_ref["all_modes"].items():
    marker = " <-- GOVERNS" if mode == z_ref["governing_mode"] else ""
    print(f"  Mode {mode:4s}: Z = {val:.1f} lbs{marker}")
print(f"\nReference Z = {z_ref['Z']} lbs ({z_ref['governing_mode']})")

# Adjusted connection capacity
result = conn.Z_prime(
    num_bolts=4,
    num_bolts_in_row=4,
    load_duration="normal",
    wet_service=True,
    end_distance=7.5 * units("in"),    # 10D
    edge_distance=3.0 * units("in"),   # 4D
    bolt_spacing=4.0 * units("in"),    # 5.3D
)

print(f"\nAdjusted Connection Capacity (4 bolts):")
print(f"  Z' = {result['Z_prime']:.0f}")
print(f"  Factors:")
for name, val in result["factors"].items():
    print(f"    {name:8s} = {val:.3f}")

# 10. Combined Bending and Compression

**Problem**: Check a 6x8 Douglas Fir-Larch No. 1 member acting as a
beam-column under axial compression of 5000 lbs and bending moment of
8000 ft-lbs. Effective length = 8 ft, unbraced length = 8 ft.

NDS Equation 3.9-3 (combined bending + compression interaction):

$$\left(\frac{f_c}{F'_c}\right)^2 + \frac{f_b}{F'_b \left(1 - \frac{f_c}{F_{cE}}\right)} \leq 1.0$$

In [ ]:
# Combined bending + compression check
member = LumberSection(6, 8, "Douglas Fir-Larch", "No. 1")
P = 5000 * units("lbf")
M = 8000 * units("lbf*ft")
le = 8 * units("ft")
lu = 8 * units("ft")

print(f"Section: {member}")
print(f"P = {P}, M = {M}")
print(f"le = lu = {le}")

check_comb = WoodMemberCheck(member, method="ASD")
result = check_comb.check_combined_bending_compression(
    P=P, M=M, le_strong=le, lu=lu
)

print(f"\nResults:")
print(f"  Compression term: (fc/Fc')^2 = {result['compression_term']}")
print(f"  Bending term:     fb/(Fb'*(1 - fc/FcE)) = {result['bending_term']}")
print(f"  FcE = {result['FcE']:.1f}")
print(f"  Interaction ratio = {result['interaction_ratio']}")
print(f"  Status: {result['status']}")

print(f"\n--- Full Summary ---")
check_comb.summary()